In [1]:
print('hello')

hello


In [ ]:
import pandas as pd
import numpy as np

def remuestrear(df, col_nombre, t_muestreo):
    columna_objetivo = col_nombre

    # 2. Creamos una máscara para filtrar las celdas:
    #    - Que el índice no sea múltiplo de t_muestreo
    #    - Que el valor no sea 0
    mascara = (df.index % t_muestreo != 0) & (df[columna_objetivo] != 0)

    # 3. Obtenemos los índices originales que cumplen la condición
    indices_originales = df.index[mascara]

    # 4. Calculamos el múltiplo de t_muestreo más cercano para cada índice original.
    #    Dividimos entre t_muestreo, redondeamos al entero más cercano y multiplicamos por t_muestreo.
    indices_nuevos = (np.round(indices_originales / t_muestreo) * t_muestreo).astype(int)

    # 5. Desplazamos los valores
    for idx_orig, idx_nuevo in zip(indices_originales, indices_nuevos):
        
        # Nos aseguramos de que el nuevo índice exista en el DataFrame
        # (por si el redondeo se pasa del límite del DataFrame)
        if idx_nuevo not in df.index:
            df.loc[idx_nuevo] = 0 # O crea la fila si lo prefieres
            
        # Sumamos el valor a la celda destino (por si hay varios valores que van a la misma celda)
        df.at[idx_nuevo, columna_objetivo] += df.at[idx_orig, columna_objetivo]
        
        # Ponemos la celda original a 0 (ya que hemos desplazado el valor)
        df.at[idx_orig, columna_objetivo] = 0

    return df

In [ ]:
datos = {
    'valores': [0, 10, 0, 0, 20, 0, 0, 30, 0, 0, 0, 40, 0]
}
df = pd.DataFrame(datos)

print("DataFrame Original:")
print(df)

salida = remuestrear(df, 'valores', 5)
print("\nDataFrame con los valores desplazados:")
print(salida)

DataFrame Original:
    valores
0         0
1        10
2         0
3         0
4        20
5         0
6         0
7        30
8         0
9         0
10        0
11       40
12        0

DataFrame con los valores desplazados:
    valores
0        10
1         0
2         0
3         0
4         0
5        50
6         0
7         0
8         0
9         0
10       40
11        0
12        0


In [2]:
import pickle
datos = dict()
for i in range(1,10):
    nombre_archivo = f"../Simulaciones/sim_{i}_data.pkl"

    with open(nombre_archivo, 'rb') as f:
        datos_recuperados = pickle.load(f)
    datos[i]=datos_recuperados
    print(type(datos_recuperados),len(datos_recuperados))

<class 'dict'> 20
<class 'dict'> 20
<class 'dict'> 20
<class 'dict'> 20
<class 'dict'> 20
<class 'dict'> 20
<class 'dict'> 20
<class 'dict'> 20
<class 'dict'> 20


In [3]:
def estadisticas_rango(df, nombre_col = 'gluc_mg'):
    est =pd.DataFrame()
    columna = nombre_col

    # 2. Definimos los límites de los rangos (bins)
    # Esto creará los rangos: (0-18], (18-35], (35-60], (60-100]
    limites = [0, 69, 180, 700]

    # 3. (Opcional) Definimos etiquetas para que el resultado sea más fácil de leer
    nombres_rangos = ['TBR', 'TIR','TAR']

    # 4. Clasificamos los valores de la columna en los rangos usando pd.cut()
    est['Tiempo_en_rango'] = pd.cut(df[columna], bins=limites, labels=nombres_rangos)

    # 5. Calculamos el porcentaje
    # normalize=True devuelve proporciones (de 0 a 1). Multiplicamos por 100 para porcentaje.
    porcentajes = est['Tiempo_en_rango'].value_counts(normalize=True) * 100

    porcentajes_ok = {'TIR':[round(porcentajes['TIR'],2)],
                      'TAR':[round(porcentajes['TAR'],2)],
                      'TBR':[round(porcentajes['TBR'],2)]}
    return porcentajes_ok

estadisticas_rango(datos[1]['p_2'])


{'TIR': [27.99], 'TAR': [72.01], 'TBR': [0.0]}

In [5]:
est = pd.DataFrame(columns=['Simulacion','Paciente', 'TBR', 'TIR', 'TAR'])
# display(est)
for simu in range(1,10):
    for p in range(20):
        est_l = estadisticas_rango(datos[simu][f'p_{p}'])
        est_l.update({'Simulacion': [simu], 'Paciente': [p]})
        df_l = pd.DataFrame(est_l)
        est = pd.concat([est, df_l])
# est.head(20)

C:\Users\Usuario\AppData\Local\Temp\ipykernel_28096\4009653070.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  est = pd.concat([est, df_l])


In [7]:
for i in range(1,10):
    display(est[est['Simulacion']==i])

,Simulacion,Paciente,TBR,TIR,TAR
0,1,0,0.0,81.41,18.59
0,1,1,0.0,98.31,1.69
0,1,2,0.0,27.99,72.01
0,1,3,0.0,24.59,75.41
0,1,4,0.0,1.64,98.36
0,1,5,0.0,100.00,0.00
0,1,6,0.0,91.11,8.89
0,1,7,0.0,99.83,0.17
0,1,8,0.0,3.43,96.57
0,1,9,0.0,1.92,98.08


,Simulacion,Paciente,TBR,TIR,TAR
0,2,0,0.0,80.74,19.26
0,2,1,0.0,98.09,1.91
0,2,2,0.0,28.67,71.33
0,2,3,0.0,20.40,79.60
0,2,4,0.0,1.99,98.01
0,2,5,0.0,100.00,0.00
0,2,6,0.0,91.54,8.46
0,2,7,0.0,98.67,1.33
0,2,8,0.0,3.82,96.18
0,2,9,0.0,2.74,97.26


,Simulacion,Paciente,TBR,TIR,TAR
0,3,0,0.0,94.93,5.07
0,3,1,0.0,99.31,0.69
0,3,2,0.0,29.18,70.82
0,3,3,0.0,29.94,70.06
0,3,4,0.0,2.13,97.87
0,3,5,0.0,100.00,0.00
0,3,6,0.0,91.14,8.86
0,3,7,0.0,99.70,0.30
0,3,8,0.0,3.65,96.35
0,3,9,0.0,1.81,98.19


,Simulacion,Paciente,TBR,TIR,TAR
0,4,0,0.0,77.56,22.44
0,4,1,0.0,97.69,2.31
0,4,2,0.0,27.85,72.15
0,4,3,0.0,18.48,81.52
0,4,4,0.0,2.25,97.75
0,4,5,0.0,100.00,0.00
0,4,6,0.0,91.60,8.40
0,4,7,0.0,99.56,0.44
0,4,8,0.0,3.47,96.53
0,4,9,0.0,3.06,96.94


,Simulacion,Paciente,TBR,TIR,TAR
0,5,0,0.0,76.95,23.05
0,5,1,0.0,97.20,2.80
0,5,2,0.0,28.65,71.35
0,5,3,0.0,18.16,81.84
0,5,4,0.0,1.64,98.36
0,5,5,0.0,100.00,0.00
0,5,6,0.0,91.39,8.61
0,5,7,0.0,98.62,1.38
0,5,8,0.0,4.94,95.06
0,5,9,0.0,2.95,97.05


,Simulacion,Paciente,TBR,TIR,TAR
0,6,0,0.0,90.89,9.11
0,6,1,0.0,99.08,0.92
0,6,2,0.0,29.20,70.80
0,6,3,0.0,30.28,69.72
0,6,4,0.0,2.84,97.16
0,6,5,0.0,100.00,0.00
0,6,6,0.0,90.24,9.76
0,6,7,0.0,99.01,0.99
0,6,8,0.0,6.88,93.12
0,6,9,0.0,2.66,97.34


,Simulacion,Paciente,TBR,TIR,TAR
0,7,0,0.0,77.72,22.28
0,7,1,0.0,92.95,7.05
0,7,2,0.0,29.90,70.10
0,7,3,0.0,14.31,85.69
0,7,4,0.0,6.27,93.73
0,7,5,0.0,100.00,0.00
0,7,6,0.0,90.37,9.63
0,7,7,0.0,98.90,1.10
0,7,8,0.0,5.24,94.76
0,7,9,0.0,1.97,98.03


,Simulacion,Paciente,TBR,TIR,TAR
0,8,0,0.0,83.32,16.68
0,8,1,0.0,97.86,2.14
0,8,2,0.0,26.77,73.23
0,8,3,0.0,17.34,82.66
0,8,4,0.0,4.49,95.51
0,8,5,0.0,100.00,0.00
0,8,6,0.0,90.65,9.35
0,8,7,0.0,97.69,2.31
0,8,8,0.0,11.15,88.85
0,8,9,0.0,2.89,97.11


,Simulacion,Paciente,TBR,TIR,TAR
0,9,0,0.0,85.83,14.17
0,9,1,0.0,97.77,2.23
0,9,2,0.0,29.22,70.78
0,9,3,0.0,23.56,76.44
0,9,4,0.0,4.57,95.43
0,9,5,0.0,100.00,0.00
0,9,6,0.0,92.03,7.97
0,9,7,0.0,98.81,1.19
0,9,8,0.0,7.10,92.90
0,9,9,0.0,4.71,95.29
